In [0]:
spark.sql("DROP TABLE IF EXISTS md_bronze.bronze_customer_CDC")
bronze_df = (
    spark.read
         .option("header", True)
         .option("inferSchema", True)
         .csv("dbfs:/FileStore/customer/customer_01082026.csv")
)

bronze_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("md_bronze.customer_deltamerge")

silver_df = (
    bronze_df)
silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("md_silver.customer_deltamerge")


gold_df = (
    bronze_df)
gold_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("md_gold.dim_customer_dm")



In [0]:
bronze_df = (
    spark.read
         .option("header", True)
         .option("inferSchema", True)
         .csv("dbfs:/FileStore/customer/customer_02082026.csv")
)

bronze_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("md_bronze.customer_updates")

silver_df = (
    bronze_df)
silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("md_silver.customer_updates")


In [0]:
%sql
MERGE INTO md_gold.dim_customer_dm t
USING md_silver.customer_updates s
ON t.customerid = s.customerid

WHEN MATCHED THEN
UPDATE SET
    t.customername = s.customername,
    t.city       = s.city,
    t.CustomerGroup =s.CustomerGroup

WHEN NOT MATCHED THEN
INSERT (
    customerid,
    customername,
    city,
    CustomerGroup
)
VALUES (
    s.customerid,
    s.customername,
    s.city,
    s.CustomerGroup
)

In [0]:
%sql
select *
from md_gold.dim_customer_dm

In [0]:
from pyspark.sql.functions import md5, concat_ws

silver_df = (
    bronze_df
        .withColumn(
            "hash_key",
            md5(
                concat_ws(
                              "|",
                "CustomerName",
                "City",
                "CustomerGroup"
                )
            )
        )
)

